# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR² clinical dataset using the `mlcroissant` library. The exploration focuses on proper use of Croissant `@id` fields for data referencing.

### Dataset Source
The dataset is described by a Croissant schema located at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset's metadata and discover what is available.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Create a Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# Access top-level dataset metadata as a dict
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview

Below, we programmatically discover available record sets (tables), their `@id`s, and their field/column structure for reference using mlcroissant's metadata inspection tools.

In [ ]:
# List all record sets by their `@id`
record_sets = [r for r in dataset.metadata.record_sets]
print('Available RecordSets (@id, name):')
for rec in record_sets:
    rec_id = rec['@id']
    rec_name = rec.get('name', '')
    print(f"- @id: {rec_id}  name: {rec_name}")
    # List the fields/columns of the record set
    print('  Columns/Fields:')
    for f in rec.get('fields', []):
        field_id = f['@id']
        field_name = f.get('name','')
        print(f"    - @id: {field_id}  name: {field_name}")
    print()

## 3. Data Extraction

Using the discovered record set `@id`s, we extract data into pandas DataFrames for analysis.

For this dataset, there is likely a main record set representing the clinical table. We'll auto-detect all record sets.

In [ ]:
# Extract data from all available record sets
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"  Columns: {list(dataframes[rs_id].columns)}")
    print(f"  Number of records: {len(dataframes[rs_id])}\n")
# Select one main record set for further EDA
# If there is only one record set, use it; otherwise, select the largest
if len(record_set_ids)==1:
    main_rs_id = record_set_ids[0]
else:
    # Choose the one with most records (could be 'clinical_table', etc.)
    main_rs_id = max(record_set_ids, key=lambda rs: len(dataframes[rs]))
print(f"Selected main RecordSet: {main_rs_id}")
print("Sample of columns:", dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

We demonstrate numeric filtering, normalizing, and grouping on the main table. All field/column access is done by `@id` as discovered above.

- Select a numeric field (such as age or diagnosis interval).
- Filter records with a value above a threshold.
- Normalize the field.
- Group records by another attribute (such as 'Sex' or 'MSI Status').

In [ ]:
# For illustration, select likely numeric fields by their @id
# Let's auto-detect a numeric field: choose an int/float column with sufficient variance
main_df = dataframes[main_rs_id]
numeric_field_id = None
for col in main_df.columns:
    if pd.api.types.is_numeric_dtype(main_df[col]):
        if main_df[col].nunique()>5: # avoid pure binary/id fields
            numeric_field_id = col
            break
if numeric_field_id is None:
    numeric_field_id = main_df.select_dtypes('number').columns[0]
print(f"Using numeric field (by @id): {numeric_field_id}")
# Choose a threshold for demo
threshold = main_df[numeric_field_id].mean()
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Find a likely group field (categorical with 2-10 unique values)
group_field_id = None
for col in main_df.columns:
    if main_df[col].dtype==object and 2 <= main_df[col].nunique() <= 10:
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization

We plot the distribution of the selected numeric field and the grouped means if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(6,4))
sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()
if group_field_id:
    plt.figure(figsize=(6,4))
    sns.barplot(x=grouped_df.index, y=grouped_df[numeric_field_id])
    plt.title(f'Mean of {numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion

This notebook demonstrates programmatic exploration and processing of the FAIR² clinical oncology dataset using `mlcroissant`. All record sets, fields, and columns were referenced strictly by their `@id`. You can adapt this workflow for further clinical analyses or FAIR data pipelines.